In [2]:
# Libraries.
import re
import os
import glob
import time
import requests
import pandas as pd
import pyarrow.parquet as pq

In [2]:
# Gets the UberonID of tissues.
def getUberonID(tissueType):
    baseURL = "https://www.ebi.ac.uk/ols4/api"
    params = {"q": tissueType, "ontology": "uberon", "exact": True}

    try:
        response = requests.get(f"{baseURL}/search", params=params)
        response.raise_for_status()
        data = response.json()
        
        docs = data.get("response", {}).get("docs", [])
        if docs:
            return docs[0].get("obo_id")

    except requests.exceptions.RequestException as e:
        print(f"Error connecting to OLS4 API: {e}")
    return None

In [12]:
def getTissue(uberonID):
    baseURL = "https://www.ebi.ac.uk/ols4/api"
    params = {"q": uberonID, "ontology": "uberon", "exact": True}

    try:
        response = requests.get(f"{baseURL}/search", params=params)
        response.raise_for_status()
        data = response.json()
        
        docs = data.get("response", {}).get("docs", [])
        if docs:
            return docs[0].get("label")

    except requests.exceptions.RequestException as e:
        print(f"Error connecting to OLS4 API: {e}")
    return None

In [4]:
sharedUberonID = [
"UBERON:0000473",
"UBERON:0000945",
"UBERON:0000948",
"UBERON:0000955",
"UBERON:0000956",
"UBERON:0000966",
"UBERON:0000992",
"UBERON:0001003",
"UBERON:0001043",
"UBERON:0001155",
"UBERON:0001264",
"UBERON:0001891",
"UBERON:0001898",
"UBERON:0002037",
"UBERON:0002048",
"UBERON:0002106",
"UBERON:0002107",
"UBERON:0002108",
"UBERON:0002113",
"UBERON:0002114",
"UBERON:0002240",
"UBERON:0002367",
"UBERON:0002370"
]


In [5]:
sharedTissues = [getTissue(uberonID) for uberonID in sharedUberonID]
sharedTissues

['testis',
 'stomach',
 'heart',
 'brain',
 'cerebral cortex',
 'retina',
 'ovary',
 'skin epidermis',
 'esophagus',
 'colon',
 'pancreas',
 'midbrain',
 'hypothalamus',
 'cerebellum',
 'lung',
 'spleen',
 'liver',
 'small intestine',
 'kidney',
 'duodenum',
 'spinal cord',
 'prostate gland',
 'thymus']

In [6]:
martExport = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/mart_export.txt", sep="\t")
martExport = martExport.dropna(subset=["Mouse gene stable ID"])
humanGenesAll = martExport["Gene stable ID"].str.upper().str.strip()
mouseGenesAll = martExport["Mouse gene stable ID"].str.upper().str.strip()

# Identifying Commmon Tissues 

In [ ]:
# Processing for Tabula Muris.
# filePath = "/Users/andrewhsu/Projects/McNair/data/Tabula-Muris/GSE132040_MACA_Bulk_metadata.csv"
# acronymMap = {
#     "BAT": "Brown Adipose Tissue",
#     "SCAT": "Subcutaneous Adipose Tissue",
#     "GAT": "Gonadal Adipose Tissue",
#     "WBC": "White Blood Cells",
#     "MAT": "Marrow Adipose Tissue",
#     "NA1": "NA"
# }

# tabulaMurisMetadata = pd.read_csv(filePath)
# tissues = tabulaMurisMetadata["source name"].apply(lambda x: str(x)[:str(x).find("_")])
# tissues[tissues.str.isupper()] =  tissues[tissues.str.isupper()].map(acronymMap)
# tabulaMurisMetadata["source name"] = tissues

# tabulaMurisMetadata.to_csv("/Users/andrewhsu/Projects/McNair/data/Tabula-Muris/GSE132040_MACA_Bulk_metadata.csv", sep=",", index=False)

In [61]:
# Get tissue and Uberon ID mappings.
tissueFile = "/Users/andrewhsu/Projects/McNair/notes/researchNotes.txt"
uberonID = []
tissues = []
fileName = None
fileUberonIDMap = {}
fileTissueMap = {}
with open(tissueFile, "r") as f:
    for line in f:
        line = line.strip()
        if ":" in line and "Gene Format" not in line:
            if fileName is not None:
                fileUberonIDMap[fileName] = uberonID
                fileTissueMap[fileName] = tissues
            fileName = line[:line.find(" ")]
            tissues = []
            uberonID = []
        elif not line or "Gene Format" in line:
            continue
        else:
            tissues.append(line)
            uberonID.append(getUberonID(line))
if fileName is not None:
    fileUberonIDMap[fileName] = uberonID
    fileTissueMap[fileName] = tissues

In [62]:
maxTissues = 0
maxHumanFile = None
maxMouseFile = None
maxUberonList = None

humanFiles = ["GTEx", "HPA", "E-MTAB-1733"] 
mouseFiles = ["MGI", "E-MTAB-6081", "Tabula-Muris"]
for humanFile in humanFiles:
    humanFileUberons = set(fileUberonIDMap[humanFile])
    for mouseFile in mouseFiles:
        mouseFileUberons = set(fileUberonIDMap[mouseFile])
        sharedUberon = humanFileUberons.intersection(mouseFileUberons)
        numSharedTissues = len(sharedUberon)
        if numSharedTissues > maxTissues:
            maxTissues = numSharedTissues
            maxHumanFile = humanFile
            maxMouseFile = mouseFile
            maxUberonList = sharedUberon

tissueList = [getTissue(uberon) for uberon in maxUberonList]
print(f"{maxHumanFile} and {maxMouseFile} had {maxTissues} tissues.")
print(f"Tissues: {tissueList}")

GTEx and E-MTAB-6081 had 8 tissues.
Tissues: ['stomach', 'colon', 'brain', 'esophagus', 'pancreas', 'kidney', 'heart', 'liver']


In [18]:
a = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/GTEx/humanData.parquet")
a

,ENSG00000290825.2,ENSG00000223972.6,ENSG00000310526.1,ENSG00000243485.6,ENSG00000237613.3,ENSG00000308361.1,ENSG00000290826.2,ENSG00000268020.3,ENSG00000186092.7,ENSG00000241860.8,...,ENSG00000210176.1,ENSG00000210184.1,ENSG00000210191.1,ENSG00000198786.2,ENSG00000198695.2,ENSG00000210194.1,ENSG00000198727.2,ENSG00000210195.2,ENSG00000210196.2,Tissue.Type
GTEX-1117F-1426-SM-H65ZH,0.000000,0.0,12.608533,0.000000,0.000000,0.0,0.000000,0.000000,0.053791,0.070991,...,0.000000,0.000000,0.000000,2028.000732,1050.950928,0.000000,11504.819336,0.000000,1.035472,Thyroid
GTEX-111CU-0226-SM-5GZXC,0.044880,0.0,1.878017,0.077797,0.048562,0.0,0.066231,0.000000,0.016166,0.059081,...,0.000000,2.151955,1.192163,12385.191406,16566.837891,46.001930,25838.982422,0.641239,0.000000,Thyroid
GTEX-111FC-1026-SM-5GZX1,0.000000,0.0,3.470647,0.000000,0.029226,0.0,0.159439,0.000000,0.038916,0.102719,...,3.691368,0.863405,0.717477,2713.388916,1756.829590,3.691368,17877.078125,0.000000,2.996522,Thyroid
GTEX-111VG-0526-SM-5N9BW,0.000000,0.0,2.463233,0.000000,0.000000,0.0,0.000000,0.050883,0.031447,0.041503,...,0.000000,0.697705,0.000000,5965.480957,3862.022949,3.579530,27664.302734,0.000000,2.421447,Thyroid
GTEX-111YS-0726-SM-5GZY8,0.027568,0.0,1.945967,0.000000,0.000000,0.0,0.000000,0.000000,0.019860,0.076615,...,0.000000,0.000000,0.000000,7271.621582,7534.889648,15.824091,31307.343750,0.000000,0.000000,Thyroid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GTEX-T6MO-1026-SM-4DM72,0.033553,0.0,7.076979,0.000000,0.036306,0.0,0.099032,0.000000,0.000000,0.068710,...,0.000000,0.000000,0.000000,9251.464844,14219.177734,36.684940,19934.179688,3.835244,0.930611,Fallopian Tube
GTEX-TML8-1026-SM-GPRWP,0.000000,0.0,6.042776,0.000000,0.000000,0.0,0.050594,0.000000,0.000000,0.112830,...,0.000000,0.000000,0.000000,9241.834961,13979.801758,44.980282,16974.019531,0.489842,0.950870,Fallopian Tube
GTEX-TMMY-1826-SM-HL9U6,0.000000,0.0,3.569392,0.000000,0.018686,0.0,0.050970,0.000000,0.049763,0.045467,...,0.000000,0.000000,0.000000,3366.389893,5807.969238,14.632805,14623.607422,0.493482,0.478968,Fallopian Tube
GTEX-TSE9-2326-SM-EZ6ME,0.040176,0.0,4.542538,0.034821,0.043472,0.0,0.000000,0.046830,0.000000,0.126344,...,0.000000,0.000000,0.000000,2931.179932,1772.325073,1.647201,13402.508789,0.000000,2.228566,Fallopian Tube


In [7]:
# Determine how many tissues human and mouse datasets share (23 total tissues).
human = set(fileUberonIDMap["GTEx"] + fileUberonIDMap["HPA"] + fileUberonIDMap["E-MTAB-1733"])
mouse = set(fileUberonIDMap["MGI"] + fileUberonIDMap["E-MTAB-6081"] + fileUberonIDMap["Tabula-Muris"])
human.intersection(mouse)

{'UBERON:0000473',
 'UBERON:0000945',
 'UBERON:0000948',
 'UBERON:0000955',
 'UBERON:0000956',
 'UBERON:0000966',
 'UBERON:0000992',
 'UBERON:0001003',
 'UBERON:0001043',
 'UBERON:0001155',
 'UBERON:0001264',
 'UBERON:0001891',
 'UBERON:0001898',
 'UBERON:0002037',
 'UBERON:0002048',
 'UBERON:0002106',
 'UBERON:0002107',
 'UBERON:0002108',
 'UBERON:0002113',
 'UBERON:0002114',
 'UBERON:0002240',
 'UBERON:0002367',
 'UBERON:0002370'}

In [6]:
human = set(fileTissueMap["GTEx"] + fileTissueMap["HPA"] + fileTissueMap["E-MTAB-1733"])
mouse = set(fileTissueMap["MGI"] + fileTissueMap["E-MTAB-6081"] + fileTissueMap["Tabula-Muris"])
human.intersection(mouse) 

{'brain',
 'cerebellum',
 'cerebral cortex',
 'colon',
 'duodenum',
 'esophagus',
 'heart',
 'hypothalamus',
 'kidney',
 'liver',
 'lung',
 'midbrain',
 'ovary',
 'pancreas',
 'retina',
 'skin',
 'small intestine',
 'spinal cord',
 'spleen',
 'stomach',
 'testis',
 'thymus'}

# Processing GTEx

In [ ]:
# Turn raw human data into a data frame with genes as columns and patient samples as rows.
humanDataRaw = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/GTEx_Analysis_2025-08-22_v11_RNASeQCv2.4.3_gene_tpm.parquet") # Dataset containing the TPM values for each sample and gene.
humanMetaData = pd.read_csv("https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt", sep="\t") # Metadata for each sample, contains information on the tissue type.

humanDataArray = []
tissueTypes = humanMetaData["SMTS"].unique()
for idx, tissueType in enumerate(tissueTypes):
    # Extract samples that are expressed in tissueType.
    samplesWithTissue = humanMetaData[humanMetaData["SMTS"] == tissueType]
    sampleIDs = samplesWithTissue["SAMPID"]

    # Extract the TPMs value from humanDataRaw.
    expressionLevels = [humanDataRaw[sampleID].to_numpy() for sampleID in sampleIDs if sampleID in humanDataRaw.columns]

    # Turn the TPM array into a dataframe and format the rows and columns.
    if(len(expressionLevels) != 0):
        sampleDF = pd.DataFrame(expressionLevels)
        sampleDF.columns = humanDataRaw.index.to_numpy()
        sampleDF.index = [sampleID for sampleID in sampleIDs if sampleID in humanDataRaw.columns]
        sampleDF["Tissue.Type"] = tissueType
        
        humanDataArray.append(sampleDF)
        print(f"Completed {tissueType.title()} Dataset ({idx + 1})")

humanDataDF = pd.concat(humanDataArray)
humanDataDF.to_parquet("/Users/andrewhsu/Projects/McNair/data/humanData.parquet")
humanDataDF.to_csv("/Users/andrewhsu/Projects/McNair/data/humanData.csv")

/var/folders/w9/_9xtsnhn18g346dc_nzw211c0000gn/T/ipykernel_3381/356577532.py:2: DtypeWarning: Columns (0: SMGTC) have mixed types. Specify dtype option on import or set low_memory=False.
  humanMetaData = pd.read_csv("https://storage.googleapis.com/adult-gtex/annotations/v11/metadata-files/GTEx_Analysis_v11_Annotations_SampleAttributesDS.txt", sep="\t") # Metadata for each sample, contains information on the tissue type.


Completed Thyroid Dataset (1)
Completed Blood Vessel Dataset (2)
Completed Muscle Dataset (3)
Completed Skin Dataset (4)
Completed Adrenal Gland Dataset (5)
Completed Pancreas Dataset (6)
Completed Blood Dataset (7)
Completed Brain Dataset (8)
Completed Adipose Tissue Dataset (9)
Completed Heart Dataset (10)
Completed Lung Dataset (11)
Completed Kidney Dataset (12)
Completed Ovary Dataset (13)
Completed Uterus Dataset (14)
Completed Vagina Dataset (15)
Completed Colon Dataset (16)
Completed Breast Dataset (17)
Completed Salivary Gland Dataset (18)
Completed Spleen Dataset (19)
Completed Esophagus Dataset (20)
Completed Stomach Dataset (21)
Completed Small Intestine Dataset (22)
Completed Prostate Dataset (23)
Completed Testis Dataset (24)
Completed Nerve Dataset (25)
Completed Liver Dataset (26)
Completed Pituitary Dataset (27)
Completed Bladder Dataset (28)
Completed Cervix Uteri Dataset (29)
Completed Fallopian Tube Dataset (30)


In [20]:
gtex = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/GTEx/humanData.parquet")
uberonID = pd.Series(gtex["Tissue.Type"].unique()).apply(lambda x: getUberonID(x))
tissueUberonMaps = dict(zip(gtex["Tissue.Type"].unique(), uberonID))
gtex["UberonID"] = gtex["Tissue.Type"].map(tissueUberonMaps)
gtexFiltered = gtex[gtex["UberonID"].isin(sharedUberonID)]
GTExGenes = [gene[:gene.find(".")].strip() for gene in gtexFiltered.columns]
filter = set(GTExGenes).intersection(set(humanGenesAll))

In [21]:
gtexFiltered2 = gtexFiltered.loc[:, list(filter)]

KeyError: "None of [Index(['ENSG00000163106', 'ENSG00000169592', 'ENSG00000112249',\n       'ENSG00000081479', 'ENSG00000007545', 'ENSG00000171490',\n       'ENSG00000140521', 'ENSG00000183560', 'ENSG00000120798',\n       'ENSG00000177685',\n       ...\n       'ENSG00000125285', 'ENSG00000113558', 'ENSG00000175356',\n       'ENSG00000140464', 'ENSG00000108950', 'ENSG00000100593',\n       'ENSG00000019991', 'ENSG00000133104', 'ENSG00000138111',\n       'ENSG00000204842'],\n      dtype='str', length=17659)] are in the [columns]"

# Processing MGI

In [ ]:
# Turn mouse data into parquet files for easier manipulation.
mouseDataArray = []
mouseFiles = glob.glob("/Users/andrewhsu/Projects/McNair/data/gxdrnaseq/*.rpt")
for idx, mouseFile in enumerate(mouseFiles):
    # Convert CSV file sinto Parquet files.
    sampleDF = pd.read_csv(mouseFile, delimiter="|")
    sampleDF.to_parquet(f"/Users/andrewhsu/Projects/McNair/data/gxdrnaseq/{os.path.basename(mouseFile).replace(".rpt", ".parquet")}", engine="pyarrow")
    print(f"Completed {os.path.basename(mouseFile)} ({idx + 1}).")


Completed E-MTAB-8964.rpt (1).
Completed E-MTAB-599.rpt (2).
Completed E-MTAB-5449.rpt (3).
Completed E-GEOD-68284.rpt (4).
Completed E-GEOD-68283.rpt (5).
Completed E-MTAB-6435.rpt (6).
Completed E-MTAB-5707.rpt (7).
Completed E-GEOD-74747.rpt (8).
Completed E-GEOD-60243.rpt (9).
Completed E-GEOD-72491.rpt (10).
Completed E-MTAB-9164.rpt (11).
Completed E-MTAB-8518.rpt (12).
Completed E-MTAB-9148.rpt (13).
Completed E-MTAB-5772.rpt (14).
Completed E-GEOD-68155.rpt (15).
Completed E-GEOD-22131.rpt (16).
Completed E-MTAB-7182.rpt (17).
Completed E-GEOD-77720.rpt (18).
Completed E-MTAB-7790.rpt (19).
Completed E-GEOD-55180.rpt (20).
Completed E-GEOD-33979.rpt (21).
Completed E-GEOD-45684.rpt (22).
Completed E-MTAB-8402.rpt (23).
Completed E-MTAB-2328.rpt (24).
Completed E-MTAB-5914.rpt (25).


# Processing E-MTAB-1733

In [123]:
emtab1733 = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/E-MTAB-1733/E-MTAB-1733-query-results.tpmss.tsv", sep="\t", skiprows=4)
filter = set(emtab1733.columns).intersection(sharedTissues)
emtab1733Filtered = emtab1733.loc[:, list(filter)]
emtab1733Filtered.to_parquet("/Users/andrewhsu/Projects/McNair/data/E-MTAB-1733/E-MTAB-1733-Filtered.parquet")

# Processing HPA

In [124]:
hpa = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/HPA/rna_tissue_consensus.tsv", sep="\t")
hpaFiltered = hpa[hpa["Tissue"].isin(sharedTissues)]
hpaFiltered.to_parquet("/Users/andrewhsu/Projects/McNair/data/HPA/hpaFiltered.parquet")

# Processing Tabula Muris

In [125]:
tabulaMurisMetadata = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/Tabula-Muris/GSE132040_MACA_Bulk_metadata.csv")
tabulaMuris = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/Tabula-Muris/GSE132040_190214_A00111_0269_AHH3J3DSXX_190214_A00111_0270_BHHMFWDSXX.csv")
sampleTissueMaps = dict(zip(tabulaMurisMetadata["Sample name"], tabulaMurisMetadata["source name"]))
filteredSampleTissueMaps = {str(k).strip() + ".gencode.vM19": str(v).lower() for k, v in sampleTissueMaps.items() if str(v).lower() in sharedTissues}
tabulaMurisFiltered = tabulaMuris.loc[:, list(filteredSampleTissueMaps.keys())]
tabulaMurisFiltered.to_parquet("/Users/andrewhsu/Projects/McNair/data/Tabula-Muris/TabulaMurisFiltered.parquet")

# Identifying threshod cutoffs.

In [ ]:
# Identify threshold cutoffs for low, medium, and high TPM levels.
thresholds = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/identifyThreshold.txt", header=None, names=["Avg_QN_TPM", "Threshold Level"])
thresholdCutoffs = thresholds.groupby("Threshold Level").agg(["min", "max"]).sort_values(("Avg_QN_TPM", "min"))
thresholdCutoffs.to_csv("/Users/andrewhsu/Projects/McNair/data/thresholdCutoffs.csv")

In [3]:
humanData = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/humanData.parquet")
samplesGrouped = humanData.groupby("Tissue.Type")

In [16]:
hpa = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/HPA/rna_tissue_consensus.tsv", sep="\t")
hpa

,Gene,Gene name,Tissue,nTPM
0,ENSG00000000003,TSPAN6,adipose tissue,20.1
1,ENSG00000000003,TSPAN6,adrenal gland,13.3
2,ENSG00000000003,TSPAN6,amygdala,8.7
3,ENSG00000000003,TSPAN6,appendix,4.6
4,ENSG00000000003,TSPAN6,basal ganglia,8.4
...,...,...,...,...
1026287,ENSG00000291317,TMEM276,thymus,5.8
1026288,ENSG00000291317,TMEM276,thyroid gland,10.9
1026289,ENSG00000291317,TMEM276,tongue,6.1
1026290,ENSG00000291317,TMEM276,tonsil,4.6


In [17]:
hpaPivot = hpa.pivot(index="Gene", columns="Tissue", values="nTPM")
hpaPivot.loc[:, list(hpaPivot.isna().sum() == 0)]

Tissue,adipose tissue,adrenal gland,appendix,bone marrow,breast,cerebral cortex,cervix,choroid plexus,colon,duodenum,...,small intestine,smooth muscle,spleen,stomach,testis,thymus,thyroid gland,tongue,tonsil,urinary bladder
Gene,,,,,,,,,,,,,,,,,,,,,
ENSG00000000003,20.1,13.3,4.6,0.6,24.0,6.6,20.8,73.2,30.7,18.5,...,12.7,6.7,6.1,13.9,65.7,3.9,18.1,5.0,6.3,30.1
ENSG00000000005,22.3,0.2,0.6,0.0,14.2,0.2,0.5,0.0,0.7,0.0,...,0.4,0.7,0.1,0.1,0.1,0.3,0.2,5.4,0.1,0.2
ENSG00000000419,37.2,48.7,31.4,50.2,37.5,24.2,30.2,27.2,35.0,33.0,...,35.5,34.0,33.0,34.0,36.4,39.9,43.4,51.2,48.6,47.4
ENSG00000000457,5.3,5.5,7.0,3.8,10.2,3.8,6.5,5.3,7.3,7.2,...,6.5,5.8,7.1,7.3,6.2,9.1,7.1,8.5,9.2,7.0
ENSG00000000460,1.7,1.1,3.1,8.5,2.7,1.0,2.1,0.6,2.4,1.1,...,1.7,2.5,2.6,1.8,9.6,10.8,1.6,1.4,7.4,2.9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000291313,2.1,0.0,0.0,0.1,0.0,1.0,0.0,0.0,0.9,0.0,...,0.0,0.0,0.4,1.3,0.0,0.0,0.0,0.0,0.0,0.0
ENSG00000291314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.2,0.0,0.0,0.0,0.0,0.1,0.2,0.0,0.4
ENSG00000291315,0.0,0.0,2.4,0.0,0.0,0.0,0.0,0.0,1.2,0.0,...,1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
rptFiles = glob.glob("/Users/andrewhsu/Projects/McNair/data/gxdrnaseq/*.rpt")
allDFs = (pd.read_csv(rptFile, sep="|") for rptFile in rptFiles)
mgiDF = pd.concat(allDFs, ignore_index=True)

In [22]:
set(hpaPivot.columns[1:]).intersection(set(mgiDF[mgiDF["Age"] == "postnatal adult"]["Anatomical Structure"].unique()))

{'liver', 'lung', 'spleen', 'thymus'}

In [52]:
mgiDFAdultOnly = mgiDF[mgiDF["Age"] == "postnatal adult"]
mgiDFAdultOnly = mgiDFAdultOnly.groupby(["Ensembl ID", "Anatomical Structure"])["avg_qnTPM"].mean().reset_index()
mgiDFAdultOnly = mgiDFAdultOnly.pivot(index="Ensembl ID", columns="Anatomical Structure", values="avg_qnTPM")
mgiDFAdultOnly

Anatomical Structure,heart,hippocampus,liver,lung,spleen,thymus
Ensembl ID,,,,,,
ENSMUSG00000000001,40.0,66.0,136.0,231.0,223.0,428.0
ENSMUSG00000000003,0.0,0.0,0.0,0.0,0.1,0.0
ENSMUSG00000000028,2.0,2.0,1.0,3.0,20.0,68.0
ENSMUSG00000000031,8.0,0.2,0.4,2.0,0.6,7.0
ENSMUSG00000000037,0.3,0.9,0.0,0.5,1.0,3.0
...,...,...,...,...,...,...
ENSMUSG00000121629,0.0,0.7,0.0,0.0,0.6,0.1
ENSMUSG00000121651,0.0,0.0,0.0,0.1,0.5,0.2
ENSMUSG00000121816,0.1,0.1,2.0,0.2,0.5,0.5


In [3]:
gtex = pd.read_parquet("/Users/andrewhsu/Projects/McNair/data/GTEx/humanData.parquet")

In [3]:
mask = (gtex == 0).sum() / len(gtex) < 0.9
gtexFiltered = gtex.loc[:, mask]

In [ ]:
len({colName[:colName.find(".")] for colName in gtexFiltered.columns})

52790

In [4]:
emtab6081 = pd.read_csv("/Users/andrewhsu/Projects/McNair/data/E-MTAB-6081/mouse_tpm.txt", sep="\t")
emtab6081Filtered = emtab6081[(emtab6081 == 0).sum(axis=1) <= (len(emtab6081.columns) * 0.1)]

In [5]:
emtab6081Tissue = []
for colName in emtab6081.columns:
    tissueStart = re.search(r"[A-Z][a-z]", colName).start()
    emtab6081Tissue.append(colName[tissueStart:])
emtab6081Tissue

['Pancreas',
 'Liver',
 'Stomach',
 'Duodenum',
 'Jejunum',
 'Ileum',
 'Colon',
 'Kidney',
 'Quadriceps',
 'Thymus',
 'Heart',
 'Esophagus',
 'Brain',
 'Pancreas',
 'Liver',
 'Stomach',
 'Duodenum',
 'Jejunum',
 'Ileum',
 'Colon',
 'Kidney',
 'Quadriceps',
 'Thymus',
 'Heart',
 'Esophagus',
 'Brain',
 'Pancreas',
 'Liver',
 'Stomach',
 'Duodenum',
 'Jejunum',
 'Ileum',
 'Colon',
 'Kidney',
 'Quadriceps',
 'Thymus',
 'Heart',
 'Esophagus',
 'Brain']

In [7]:
sharedTissues = set(gtex["Tissue.Type"]).intersection(emtab6081Tissue)

In [9]:
mask = gtex["Tissue.Type"].isin(sharedTissues)
gtexFiltered = gtex.loc[mask]

In [12]:
gtexFiltered.to_parquet("/Users/andrewhsu/Projects/McNair/data/gtexData.parquet")

In [ ]:
emtab6081 = emtab6081.map(lambda x: x * 10)

In [14]:
emtab6081.to_parquet("/Users/andrewhsu/Projects/McNair/data/EMTAB6081Data.parquet")

In [21]:
sum(emtab6081.isna().sum() != 0)

0

In [18]:
sum(gtex[gtex["Tissue.Type"].isin(sharedTissues)].isna().sum() != 0)

0

In [27]:
len(gtexFiltered2.columns) * 0.1

5279.0

In [35]:
(gtexFiltered2 == 0).sum(axis=0) <= (len(gtexFiltered2.columns) * 0.9)

ENSG00000290825.2    True
ENSG00000310526.1    True
ENSG00000243485.6    True
ENSG00000237613.3    True
ENSG00000290826.2    True
                     ... 
ENSG00000210194.1    True
ENSG00000198727.2    True
ENSG00000210195.2    True
ENSG00000210196.2    True
Tissue.Type          True
Length: 52790, dtype: bool

In [37]:
gtexFiltered2.loc[:, ((gtexFiltered2 == 0).sum(axis=0) <= (len(gtexFiltered2.columns) * 0.1))]

,ENSG00000290825.2,ENSG00000310526.1,ENSG00000237613.3,ENSG00000290826.2,ENSG00000268020.3,ENSG00000186092.7,ENSG00000241860.8,ENSG00000233750.3,ENSG00000308579.1,ENSG00000268903.1,...,ENSG00000210176.1,ENSG00000210184.1,ENSG00000210191.1,ENSG00000198786.2,ENSG00000198695.2,ENSG00000210194.1,ENSG00000198727.2,ENSG00000210195.2,ENSG00000210196.2,Tissue.Type
GTEX-111CU-0526-SM-5EGHK,0.000000,0.612349,0.000000,0.000000,0.000000,0.057187,0.058056,0.020685,4.068348,0.267349,...,1.084893,0.634386,0.527166,2439.027344,2944.897705,11.933819,8588.482422,0.000000,0.000000,Pancreas
GTEX-111YS-1226-SM-5EGGJ,0.016610,0.386135,0.000000,0.049024,0.000000,0.023931,0.043731,0.017312,4.475158,0.000000,...,1.362005,0.000000,0.000000,2133.339111,2734.978027,9.534033,11147.070312,0.949276,0.921356,Pancreas
GTEX-1122O-0726-SM-5GIEV,0.000000,0.935880,0.000000,0.072210,0.000000,0.044063,0.039365,0.038250,7.523230,0.411986,...,1.003097,0.782076,0.000000,2317.819336,3608.737549,17.052656,12645.546875,0.000000,0.339283,Pancreas
GTEX-1128S-0826-SM-5GZZI,0.000000,1.389022,0.000000,0.087381,0.034509,0.031992,0.015156,0.154286,22.455864,4.187715,...,0.000000,0.000000,0.000000,733.266113,569.954651,0.404610,8317.146484,0.423002,0.000000,Pancreas
GTEX-117YX-0226-SM-5EGH6,0.000000,0.686950,0.018129,0.098904,0.039060,0.048281,0.031859,0.026195,0.883221,0.000000,...,1.373899,0.535588,0.890132,2527.486328,3684.884033,11.907127,8695.977539,0.957566,1.394104,Pancreas
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GTEX-ZVT3-1626-SM-5GU66,0.021561,1.348763,0.000000,0.063636,0.000000,0.093193,0.028383,0.000000,6.440472,1.307037,...,0.000000,4.135258,0.000000,7201.433594,5897.970215,10.018512,40513.937500,0.616112,1.195982,Liver
GTEX-ZVT4-0626-SM-5E45T,0.000000,1.051334,0.054926,0.049940,0.039446,0.097515,0.039599,0.008818,5.847215,0.000000,...,0.000000,1.081760,1.348391,5487.423340,4739.679688,6.012391,47358.128906,0.967028,0.469293,Liver
GTEX-ZYT6-0626-SM-5E45V,0.015521,0.590438,0.016794,0.000000,0.000000,0.167719,0.027243,0.032354,2.727261,0.000000,...,0.424241,0.496146,0.412290,3610.788086,3061.635010,4.666646,30968.281250,0.000000,0.860959,Liver
GTEX-ZYY3-0626-SM-5NQ6W,0.018798,2.367814,0.000000,0.055483,0.087648,0.027085,0.032995,0.000000,18.167242,1.266202,...,1.027642,0.000000,0.499347,6278.348145,5353.770996,4.110568,38080.480469,0.000000,1.564132,Liver


In [14]:
emtab6081

,199_1_Pancreas,199_2_Liver,199_3_Stomach,199_4_Duodenum,199_5_Jejunum,199_6_Ileum,199_7_Colon,199_8_Kidney,199_9_Quadriceps,199_10_Thymus,...,199_30_Duodenum,199_31_Jejunum,199_32_Ileum,199_33_Colon,199_34_Kidney,199_35_Quadriceps,199_36_Thymus,199_37_Heart,199_38_Esophagus,199_39_Brain
ENSMUSG00000000001,0.512887,5.926545,5.733410,11.268121,14.997442,18.987661,12.958968,7.782625,1.466109,18.098834,...,8.944480,13.962269,14.539717,11.311051,7.060182,1.546428,17.419960,2.094721,10.634418,3.348194
ENSMUSG00000000003,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.019166,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000000028,0.010531,0.111843,0.381710,0.642235,0.692123,1.464154,1.527913,0.264255,0.275475,6.603444,...,0.699866,1.087083,1.022826,0.992046,0.143004,0.265119,6.619279,0.266752,0.473103,0.179512
ENSMUSG00000000031,0.094048,0.009826,0.112698,0.013387,0.022707,0.034533,0.013361,0.004695,58.680567,1.525592,...,0.057532,0.154973,0.118522,0.219733,0.181899,99.628396,1.877733,1.272518,157.098081,0.151693
ENSMUSG00000000037,0.000000,0.000000,0.019612,0.026431,0.008538,0.068887,0.272336,0.038901,0.017164,0.466792,...,0.021613,0.129843,0.026193,0.422155,0.037721,0.036856,0.107689,0.060564,0.223384,0.130295
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109574,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.021082,...,0.000000,0.000000,0.000000,0.006295,0.002515,0.000000,0.007032,0.000000,0.000000,0.018783
ENSMUSG00000109575,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.002722,0.000000,0.100099
ENSMUSG00000109576,0.000000,0.000000,0.000000,0.005492,0.010355,0.000000,0.000000,0.000000,0.005321,0.013416,...,0.000000,0.004363,0.000000,0.000000,0.003856,0.004398,0.000000,0.000000,0.000000,0.000000
ENSMUSG00000109577,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [15]:
emtab6081Filtered

,199_1_Pancreas,199_2_Liver,199_3_Stomach,199_4_Duodenum,199_5_Jejunum,199_6_Ileum,199_7_Colon,199_8_Kidney,199_9_Quadriceps,199_10_Thymus,...,199_30_Duodenum,199_31_Jejunum,199_32_Ileum,199_33_Colon,199_34_Kidney,199_35_Quadriceps,199_36_Thymus,199_37_Heart,199_38_Esophagus,199_39_Brain
ENSMUSG00000000001,0.512887,5.926545,5.733410,11.268121,14.997442,18.987661,12.958968,7.782625,1.466109,18.098834,...,8.944480,13.962269,14.539717,11.311051,7.060182,1.546428,17.419960,2.094721,10.634418,3.348194
ENSMUSG00000000028,0.010531,0.111843,0.381710,0.642235,0.692123,1.464154,1.527913,0.264255,0.275475,6.603444,...,0.699866,1.087083,1.022826,0.992046,0.143004,0.265119,6.619279,0.266752,0.473103,0.179512
ENSMUSG00000000031,0.094048,0.009826,0.112698,0.013387,0.022707,0.034533,0.013361,0.004695,58.680567,1.525592,...,0.057532,0.154973,0.118522,0.219733,0.181899,99.628396,1.877733,1.272518,157.098081,0.151693
ENSMUSG00000000056,0.146390,2.057254,2.187076,2.078597,3.327456,3.066797,3.632724,3.232770,11.514155,8.425466,...,2.659989,2.729836,1.494034,3.851922,3.334194,8.544179,8.121104,7.956638,2.160916,4.882010
ENSMUSG00000000058,0.191746,0.987661,1.972950,0.320431,0.662875,1.000735,2.770129,5.989679,3.801550,0.862032,...,0.962336,0.946071,1.646082,3.967986,6.353208,6.246681,0.807871,7.766441,12.799938,6.110061
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSMUSG00000109408,0.000000,0.022542,0.056349,0.061443,0.052136,0.139920,0.155817,0.100940,0.283198,0.203370,...,0.109777,0.167190,0.063255,0.212258,0.097739,0.163197,0.240493,0.118746,0.165927,0.256547
ENSMUSG00000109428,0.040455,0.000000,0.035419,0.049429,0.018711,0.133300,0.100407,0.024145,0.010126,0.137354,...,0.099514,0.065270,0.031562,0.123145,0.070580,0.015774,0.129388,0.060564,0.050644,0.182603
ENSMUSG00000109498,0.004275,0.000000,0.012880,0.053033,0.033425,0.046700,0.032028,0.014085,0.014761,0.090292,...,0.041050,0.040314,0.009822,0.035016,0.039900,0.011982,0.067909,0.036066,0.026519,0.006895
ENSMUSG00000109511,0.203215,0.610226,1.470052,1.494887,1.855833,3.194014,5.109746,1.295120,1.930726,10.977276,...,1.899326,2.646939,1.473079,5.310385,1.211596,1.903914,10.894105,2.763303,3.099204,2.939002
